# Explore the GROW-Qual dataset!

#### Author: Edward Jones (e.r.jones@uu.nl)

This notebook provides an introduction to _GROW-Qual_, an open source large sample hydrology and groundwater quality dataset. The dataset contains groundwater level/depth and quality observations, which are also connected to static and transient environmental variables spanning the anthroposphere, atmosphere, biosphere, geosphere and hydrosphere:
- **Groundwater quality observations** from various datasets, primarily global (e.g. _GEMS_) to national databases (e.g. _Water Quality Portal_)
- **Groundwater level and depth observations** from various datasets, including daily observations processed directly from _GGMN_ and processed time-series observations from _GROW_.
- **Transient environmental variables** from multiple sources (see _Table 1_ in the manuscript)
- **Static environmental attributes** from multiple sources (see _Table 2_ in the manuscript)

Data is stored in Zarr format.

Please note, observations in _GROW-Qual.zarr_ contains all groundwater quality observations (indexed by gwms_id) and depth/level observations from _GGMN_ (indexed by ggmn_id) from 1980-2025. Processed groundwater depth/level observations from _GROW_ are contained across three zarr stores depending on timestep (GROW_d.zarr for daily; GROW_MS.zarr for monthly; GROW_YS.zarr for annual) and are indexed by grow_id.

## 1. Setup

In [ ]:
#load required modules
import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Modules loaded!")

In [ ]:
#GROW-Qual dataset
data_dir = Path("D:/GROW-Qual")
zarr_main = data_dir / "GROW-Qual.zarr"
linkages_main = data_dir / "GROW-Qual_linkages.parquet"
variable_metadata = data_dir / "GROW-Qual_zarr_variables.csv"

#GROW dataset
grow_dir = Path("D:/GROW-Qual/GROW")
zarr_grow_daily = grow_dir / "GROW_d.zarr"
zarr_grow_monthly = grow_dir / "GROW_MS.zarr"
zarr_grow_annual = grow_dir / "GROW_YS.zarr"

#plotting paths
plot_dir = data_dir / "example_plots"
os.makedirs(plot_dir, exist_ok=True)

print("Directories and files defined!")

In [ ]:
# Define plotting preferences
plt.style.use('default')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.2,
    'grid.linestyle': '-',
    'grid.linewidth': 0.5,
    'axes.linewidth': 0.5,
    'lines.linewidth': 0.5,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'figure.titlesize': 16,
    'figure.titleweight': 'bold',
})

from cartopy.crs import Globe
import cartopy.io.shapereader as shpreader

wgs84_globe = Globe(datum='WGS84', ellipse='WGS84')
WGS84_PLATECARREE = ccrs.PlateCarree(globe=wgs84_globe)
WGS84_ROBINSON = ccrs.Robinson(globe=wgs84_globe)

land_shp = shpreader.natural_earth(resolution='10m', category='physical', name='land')
land_geoms = list(shpreader.Reader(land_shp).geometries())

print("Plotting preferences assigned!")

## 2. Load datasets

In [ ]:
print("Loading main dataset (GROW-Qual.zarr)...")
ds = xr.open_zarr(zarr_main, consolidated=True)
var_metadata = pd.read_csv(variable_metadata)

print("Loading GROW groundwater level/depth datasets...")
ds_grow_daily = xr.open_zarr(zarr_grow_daily, consolidated=True)
ds_grow_monthly = xr.open_zarr(zarr_grow_monthly, consolidated=True)
ds_grow_annual = xr.open_zarr(zarr_grow_annual, consolidated=True)

print("\nLoading linkage files...")
df_main_linkages = pd.read_parquet(linkages_main)
print(df_main_linkages.head())

print("All datasets loaded successfully!")

## 3. Basic overview of _GROW-Qual_

In [ ]:
#Structure of dataset and time range
print(f"\nGROW-Qual.zarr dimensions: {dict(ds.sizes)}")
print(f"Time range: {pd.to_datetime(ds.time.values[0]).strftime('%d-%m-%Y')} to {pd.to_datetime(ds.time.values[-1]).strftime('%d-%m-%Y')}")

data_vars = list(ds.data_vars)

print("\nVariables:")
for category in var_metadata['category'].unique():
    count = len(var_metadata[var_metadata['category'] == category])
    print(f"  - {category}: {count}")

#GROW groundwater level/depth stores
for label, grow_ds in [("GROW_d (daily)", ds_grow_daily),
                        ("GROW_MS (monthly)", ds_grow_monthly),
                        ("GROW_YS (annual)", ds_grow_annual)]:
    print(f"\n{label} dimensions: {dict(grow_ds.sizes)}")
    print(f"Time range: {pd.to_datetime(grow_ds.time.values[0]).strftime('%d-%m-%Y')} to {pd.to_datetime(grow_ds.time.values[-1]).strftime('%d-%m-%Y')}")


## 4. Spatial distribution of monitoring stations

### 4.1 Global outlook

In [ ]:
#Extract station locations
print("Extracting station locations...")

#Groundwater quality monitoring stations (GROW-Qual.zarr)
gwms_lats = ds['gwms_lat'].values
gwms_lons = ds['gwms_lon'].values

#GGMN level/depth stations (GROW-Qual.zarr)
ggmn_lats = ds['ggmn_lat'].values
ggmn_lons = ds['ggmn_lon'].values

#GROW level/depth stations
grow_daily_lats = ds_grow_daily['grow_lat'].values
grow_daily_lons = ds_grow_daily['grow_lon'].values
grow_monthly_lats = ds_grow_monthly['grow_lat'].values
grow_monthly_lons = ds_grow_monthly['grow_lon'].values
grow_annual_lats = ds_grow_annual['grow_lat'].values
grow_annual_lons = ds_grow_annual['grow_lon'].values

#Groundwater quality and GGMN maps
station_types = [
    {'lons': gwms_lons, 'lats': gwms_lats, 'color': 'coral',
     'edgecolor': 'darkred', 'title': 'Groundwater quality monitoring stations', 'filename': 'gwms_stations_map.png'},
    {'lons': ggmn_lons, 'lats': ggmn_lats, 'color': 'steelblue',
     'edgecolor': 'darkblue', 'title': 'GGMN level/depth stations', 'filename': 'ggmn_stations_map.png'},
]

print("Plotting groundwater quality and GGMN level/depth stations...")
for station in station_types:
    fig = plt.figure(figsize=(16, 8))
    ax = plt.axes(projection=WGS84_ROBINSON)

    ax.set_facecolor('white')
    ax.add_geometries(land_geoms, crs=WGS84_PLATECARREE, facecolor='lightgray', edgecolor='black', linewidth=0.1, zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5, zorder=1)
    ax.add_feature(cfeature.RIVERS, linewidth=0.3, alpha=0.5, zorder=1)

    ax.scatter(station['lons'], station['lats'], c=station['color'], s=8, alpha=0.7, transform=WGS84_PLATECARREE, edgecolors=station['edgecolor'], linewidth=0.5, zorder=5)

    ax.set_global()
    ax.set_title(f"{station['title']} (n={len(station['lats'])})", fontsize=16, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, station['filename']), dpi=300, bbox_inches='tight')
    plt.show()

#GROW map
print("Plotting GROW level/depth stations by timestep...")
grow_groups = [
    {'lons': grow_annual_lons, 'lats': grow_annual_lats, 'color': 'firebrick', 'edgecolor': 'darkred', 'label': 'Annual'},
    {'lons': grow_monthly_lons, 'lats': grow_monthly_lats, 'color': 'goldenrod', 'edgecolor': 'darkgoldenrod', 'label': 'Monthly'},
    {'lons': grow_daily_lons, 'lats': grow_daily_lats, 'color': 'seagreen', 'edgecolor': 'darkgreen', 'label': 'Daily'},
]

fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=WGS84_ROBINSON)

ax.set_facecolor('white')
ax.add_geometries(land_geoms, crs=WGS84_PLATECARREE, facecolor='lightgray', edgecolor='black', linewidth=0.1, zorder=0)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5, zorder=1)
ax.add_feature(cfeature.RIVERS, linewidth=0.3, alpha=0.5, zorder=1)

for group in grow_groups:
    ax.scatter(group['lons'], group['lats'], c=group['color'], s=8, alpha=0.7,
           transform=WGS84_PLATECARREE, edgecolors=group['edgecolor'], linewidth=0.5, zorder=5, label=f"{group['label']} (n={len(group['lats'])})")

n_total = len(grow_daily_lats) + len(grow_monthly_lats) + len(grow_annual_lats)
ax.set_global()
ax.set_title(f"GROW level/depth stations (n={n_total} station-timestep entries)", fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='lower left', fontsize=11, markerscale=1.5, frameon=True, facecolor='white', framealpha=0.8)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'grow_stations_map.png'), dpi=300, bbox_inches='tight')
plt.show()

### 4.2 Regional outlook

In [ ]:
# Define bounding box and region name
#bbox = {'lon_min': -125, 'lon_max': -65, 'lat_min': 24, 'lat_max': 50}
#region_name = 'United States of America'

bbox = {'lon_min': -11, 'lon_max': 2, 'lat_min': 49.5, 'lat_max': 61}
region_name = 'The British Isles'

#bbox = {'lon_min': -31,'lon_max': 40,'lat_min': 34,'lat_max': 72}
#region_name = 'Europe'

borders_geoms = list(cfeature.BORDERS.with_scale('10m').geometries())
rivers_geoms = list(cfeature.RIVERS.with_scale('10m').geometries())

#Filter stations to bbox
def in_bbox(lons, lats):
    return (lons >= bbox['lon_min']) & (lons <= bbox['lon_max']) & (lats >= bbox['lat_min']) & (lats <= bbox['lat_max'])

gwms_in_bbox = in_bbox(gwms_lons, gwms_lats)
ggmn_in_bbox = in_bbox(ggmn_lons, ggmn_lats)
grow_daily_in_bbox = in_bbox(grow_daily_lons, grow_daily_lats)
grow_monthly_in_bbox = in_bbox(grow_monthly_lons, grow_monthly_lats)
grow_annual_in_bbox = in_bbox(grow_annual_lons, grow_annual_lats)

# Regional maps: groundwater quality + GGMN - same style as the global plot, just cropped to bbox
station_types = [
    {'lons': gwms_lons[gwms_in_bbox], 'lats': gwms_lats[gwms_in_bbox],
     'color': 'coral', 'edgecolor': 'darkred', 'title': 'Groundwater quality monitoring stations', 'filename': 'regional_gwms_stations_map.png'},
    {'lons': ggmn_lons[ggmn_in_bbox], 'lats': ggmn_lats[ggmn_in_bbox],
     'color': 'steelblue', 'edgecolor': 'darkblue', 'title': 'GGMN level/depth stations', 'filename': 'regional_ggmn_stations_map.png'},
]

print(f"Plotting groundwater quality and GGMN level/depth stations for {region_name}...")
for station in station_types:
    fig = plt.figure(figsize=(16, 8))
    ax = plt.axes(projection=WGS84_PLATECARREE)
    ax.set_extent([bbox['lon_min'], bbox['lon_max'], bbox['lat_min'], bbox['lat_max']], crs=WGS84_PLATECARREE)

    ax.set_facecolor('white')
    ax.add_geometries(land_geoms, crs=WGS84_PLATECARREE, facecolor='lightgray', edgecolor='black', linewidth=0.5, zorder=0)
    ax.add_geometries(borders_geoms, crs=WGS84_PLATECARREE, facecolor='none', edgecolor='black', linewidth=0.3, alpha=0.5, zorder=1)
    ax.add_geometries(rivers_geoms, crs=WGS84_PLATECARREE, facecolor='none', edgecolor='blue', linewidth=0.3, alpha=0.5, zorder=1)
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.geo_labels = False

    ax.scatter(station['lons'], station['lats'], c=station['color'], s=8, alpha=0.7, transform=WGS84_PLATECARREE, edgecolors=station['edgecolor'], linewidth=0.5, zorder=5)

    ax.set_title(f"{station['title']} in {region_name} (n={len(station['lats'])})", fontsize=16, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, station['filename']), dpi=300, bbox_inches='tight')
    plt.show()

# Regional GROW map - coloured by timestep, same style as global GROW plot
print(f"Plotting GROW level/depth stations by timestep for {region_name}...")
grow_groups = [
    {'lons': grow_annual_lons[grow_annual_in_bbox], 'lats': grow_annual_lats[grow_annual_in_bbox],
     'color': 'firebrick', 'edgecolor': 'darkred', 'label': 'Annual'},
    {'lons': grow_monthly_lons[grow_monthly_in_bbox], 'lats': grow_monthly_lats[grow_monthly_in_bbox],
     'color': 'goldenrod', 'edgecolor': 'darkgoldenrod', 'label': 'Monthly'},
    {'lons': grow_daily_lons[grow_daily_in_bbox], 'lats': grow_daily_lats[grow_daily_in_bbox],
     'color': 'seagreen', 'edgecolor': 'darkgreen', 'label': 'Daily'},
]

fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=WGS84_PLATECARREE)
ax.set_extent([bbox['lon_min'], bbox['lon_max'], bbox['lat_min'], bbox['lat_max']], crs=WGS84_PLATECARREE)

ax.set_facecolor('white')
ax.add_geometries(land_geoms, crs=WGS84_PLATECARREE, facecolor='lightgray', edgecolor='black', linewidth=0.5, zorder=0)
ax.add_geometries(borders_geoms, crs=WGS84_PLATECARREE, facecolor='none', edgecolor='black', linewidth=0.3, alpha=0.5, zorder=1)
ax.add_geometries(rivers_geoms, crs=WGS84_PLATECARREE, facecolor='none', edgecolor='blue', linewidth=0.3, alpha=0.5, zorder=1)
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.geo_labels = False

for group in grow_groups:
    ax.scatter(group['lons'], group['lats'], c=group['color'], s=12, alpha=0.7,
               transform=WGS84_PLATECARREE, edgecolors=group['edgecolor'], linewidth=0.5, zorder=5, label=f"{group['label']} (n={len(group['lats'])})")

n_total = sum(len(g['lats']) for g in grow_groups)
ax.set_title(f'GROW level/depth stations in {region_name} (n={n_total})', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='lower left', fontsize=11, markerscale=1.5, frameon=True, facecolor='white', framealpha=0.8)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'regional_grow_stations_map.png'), dpi=300, bbox_inches='tight')
plt.show()

## 5. Time series data (example water quality station)

In [ ]:
# Select an example GWMS station
gwms_ids = ds.coords['gwms_id'].values
example_gwms = "gwms_23297090"  #specify a station, or pick a station from gwms_ids

EXCLUDE_OUTLIERS = True  #excludes values flagged as "*" in GROW-Qual

print(f"Selected groundwater quality station: {example_gwms}")

### 5.1 Water quality data: summarise at station level and plot time series

In [ ]:
# Get all water quality parameters available for this station
wq_params = var_metadata[var_metadata['category'] == 'water_quality']['variable'].tolist()

# Extract data for parameters with observations
wq_data = {}
for param in wq_params:
    if param in ds.data_vars:
        data = ds[param].sel(gwms_id=example_gwms)

        flag_var = f"{param}_flag"
        if flag_var in ds.data_vars:
            flags = ds[flag_var].sel(gwms_id=example_gwms)
            exclude_codes = [3, 255] if EXCLUDE_OUTLIERS else [255]
            data = data.where(~flags.isin(exclude_codes))

        if np.sum(~np.isnan(data.values)) > 0:
            wq_data[param] = data

print(f"\nWater quality parameters with data: {len(wq_data)}")
for param in list(wq_data.keys())[:10]:
    n_obs = np.sum(~np.isnan(wq_data[param].values))
    print(f"  - {param}: {n_obs} observations")
if len(wq_data) > 10:
    print(f"  ... and {len(wq_data) - 10} more parameters")

In [ ]:
# Plot (up to) n_plots parameters with the highest number of observations
n_plots = 3
params_to_plot = sorted(wq_data.keys(),
                       key=lambda x: wq_data[x].to_pandas().notna().sum(),
                       reverse=True)[:n_plots]

fig, axes = plt.subplots(len(params_to_plot), 1, figsize=(16, 3*len(params_to_plot)))
if len(params_to_plot) == 1:
    axes = [axes]

for idx, param in enumerate(params_to_plot):
    data_df = wq_data[param].to_pandas()
    axes[idx].plot(data_df.index, data_df.values, 'o-', linewidth=0.8, markersize=3, alpha=0.7)
    units = wq_data[param].attrs.get('units', 'unknown')
    axes[idx].set_ylabel(f'{param.replace("qual_", "")}\n({units})')
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
fig.suptitle(f'Water quality observations ({example_gwms})', y=0.975)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'example_wq_data.png'), dpi=300, bbox_inches='tight')
plt.show()

### 5.2 Linking groundwater quality data to depth/level

In [ ]:
# Link this gwms station to its GROW and GGMN groundwater level/depth records
distance_threshold_m = 100.0
print(f"Distance threshold for matching to GROW/GGMN stations: {distance_threshold_m} m")

station_row = df_main_linkages[df_main_linkages['gwms_id'] == example_gwms]

if len(station_row) == 0:
    print(f"No linkage info found for {example_gwms}")
else:
    row = station_row.iloc[0]
    grow_id = row.get('grow_id')
    grow_distance_m = row.get('grow_distance_m')
    ggmn_id = row.get('ggmn_id')
    ggmn_distance_m = row.get('ggmn_distance_m')
    if pd.isna(ggmn_id):
        ggmn_id = row.get('ggmn_id_from_grow_id')  #fall back to the grow-derived link

    #Apply distance threshold, drop the link if the matched station exceeds this
    if pd.notna(grow_id) and (pd.isna(grow_distance_m) or grow_distance_m > distance_threshold_m):
        print(f"  Closest matched GROW station {grow_id} is {grow_distance_m:.1f} m away, exceeds distance threshold.")
        grow_id = None
    elif pd.notna(grow_id):
        print(f"  Found linked GROW station: {grow_id} (distance: {grow_distance_m:.1f} m)")

    if pd.notna(ggmn_id) and (pd.isna(ggmn_distance_m) or ggmn_distance_m > distance_threshold_m):
        print(f"  Closest matched GGMN station {ggmn_id} is {ggmn_distance_m:.1f} m away, exceeds distance threshold.")
        ggmn_id = None
    elif pd.notna(ggmn_id):
        print(f"  Found linked GGMN station: {ggmn_id} (distance: {ggmn_distance_m:.1f} m)")

    #Identify GROW variable
    grow_match = None  #(freq_label, gds, base_var, filled_var)
    if pd.notna(grow_id):
        for freq_label, gds in [('daily', ds_grow_daily), ('monthly', ds_grow_monthly), ('annual', ds_grow_annual)]:
            if grow_id not in gds.coords['grow_id'].values:
                continue
            base_vars = [
                v for v in gds.data_vars
                if v.startswith('groundwater_') and '_filled_' not in v
                and pd.api.types.is_numeric_dtype(gds[v].dtype)
            ]
            for base_var in base_vars:
                data = gds[base_var].sel(grow_id=grow_id)
                if np.sum(~np.isnan(data.values)) > 0:
                    filled_var = base_var.replace('groundwater_', 'groundwater_filled_', 1)
                    filled_var = filled_var if filled_var in gds.data_vars else None
                    grow_match = (freq_label, gds, base_var, filled_var)
                    break
            if grow_match is not None:
                break

    #Identify most monitored GGMN variable (as some stations report in multiple units)
    ggmn_best = None  #(n_obs, var_name, data)
    if pd.notna(ggmn_id):
        ggmn_vars = [
            v for v in ds.data_vars
            if 'ggmn_id' in ds[v].dims and 'time' in ds[v].dims
            and pd.api.types.is_numeric_dtype(ds[v].dtype)
        ]
        for var in ggmn_vars:
            data = ds[var].sel(ggmn_id=ggmn_id)
            n_obs = int(np.sum(~np.isnan(data.values)))
            if n_obs == 0:
                continue
            if ggmn_best is None or n_obs > ggmn_best[0]:
                ggmn_best = (n_obs, var, data)

    n_panels = int(grow_match is not None) + int(ggmn_best is not None)
    if n_panels == 0:
        print(f"No linked GROW or GGMN groundwater level/depth data found within threshold for {example_gwms}")
    else:
        fig, axes = plt.subplots(n_panels, 1, figsize=(16, 5*n_panels))
        if n_panels == 1:
            axes = [axes]
        panel = 0

        if grow_match is not None:
            freq_label, gds, base_var, filled_var = grow_match
            ax = axes[panel]; panel += 1

            if filled_var is not None:
                filled_df = gds[filled_var].sel(grow_id=grow_id).to_pandas()
                ax.plot(filled_df.index, filled_df.values, '-', color='lightgray', linewidth=1, label='Gap-filled', zorder=1)

            base_df = gds[base_var].sel(grow_id=grow_id).to_pandas()
            ax.plot(base_df.index, base_df.values, 'o', color='seagreen', markersize=3, alpha=0.8, label='Observed', zorder=2)

            units = gds[base_var].attrs.get('units', 'unknown')
            ax.set_ylabel(f'{base_var}\n({units})')
            ax.set_title(f'{grow_id} ({freq_label}; quantity station is {grow_distance_m:.1f}m away)', fontsize=12)
            ax.grid(True, alpha=0.3)
            ax.legend(loc='upper right', fontsize=9)

        if ggmn_best is not None:
            n_obs, var, data = ggmn_best
            ax = axes[panel]; panel += 1
            ggmn_df = data.to_pandas()
            ax.plot(ggmn_df.index, ggmn_df.values, 'o-', color='steelblue', linewidth=0.6, markersize=2, alpha=0.7)

            units = data.attrs.get('units', 'unknown')
            ax.set_ylabel(f'{var}\n({units})')
            ax.set_title(f'ggmn {ggmn_id} (daily; quantity station is {ggmn_distance_m:.1f}m away)', fontsize=12)
            ax.grid(True, alpha=0.3)

        axes[-1].set_xlabel('Date')

        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, 'example_groundwater_level_data.png'), dpi=300, bbox_inches='tight')
        plt.show()

### 5.3 Linking groundwater quality observations to transient environmental variables

In [ ]:
print("Checking for linked environmental data...")

#pick variables to plot
weather_vars_to_plot = {
    'precipitation_mswep_mm_year-1': {'color': '#3498db', 'ylabel': 'Precipitation\n(mm/year)', 'kind': 'bar'},
    'potential_evapotranspiration_gleam_mm_year-1': {'color': '#f39c12', 'ylabel': 'Potential ET\n(mm/year)', 'kind': 'line'},
    'air_temperature_°C': {'color': '#e74c3c', 'ylabel': '2m air temperature\n(°C)', 'kind': 'line'},
}

print("Extracting transient earth system variables for this station...")
weather_data = {}
for var, cfg in weather_vars_to_plot.items():
    if var in ds.data_vars:
        data = ds[var].sel(gwms_id=example_gwms)
        n_obs = int(np.sum(~np.isnan(data.values)))
        if n_obs > 0:
            weather_data[var] = data
            print(f"  - {var}: {n_obs} observations")
        else:
            print(f"  - {var}: no data for this station")
    else:
        print(f"  - {var}: not found in dataset")

if len(weather_data) > 0:
    fig, axes = plt.subplots(len(weather_data), 1, figsize=(16, 4*len(weather_data)))
    if len(weather_data) == 1:
        axes = [axes]

    for idx, (var, data) in enumerate(weather_data.items()):
        cfg = weather_vars_to_plot[var]
        df = data.to_pandas()
        ax = axes[idx]

        if cfg['kind'] == 'bar':
            valid_dates = df.dropna().index
            if len(valid_dates) > 1:
                median_gap_days = np.median(np.diff(valid_dates)).astype('timedelta64[D]').astype(float)
            else:
                median_gap_days = 1  # fallback for a single observation
            ax.bar(df.index, df.values, width=median_gap_days, color=cfg['color'], alpha=0.7, edgecolor='none')
            ax.grid(axis='y', alpha=0.3)
        else:
            ax.plot(df.index, df.values, color=cfg['color'])
            if 'temperature' in var:
                ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
            ax.grid(True, alpha=0.3)

        ax.set_ylabel(cfg['ylabel'])

    axes[-1].set_xlabel('Date')
    fig.suptitle(f'Earth-system variables for {example_gwms}', y=0.995)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'example_weather_data.png'), dpi=300, bbox_inches='tight')
    plt.show()
else:
    print(f"No transient earth system variable data available for {example_gwms}")

## 6. Aggregated analysis of _GROW-Qual_ (full database interactions)

### 6.1 Detailed inspection of _GROW-Qual_

#### 6.1.1 Number of observations per water quality constituent

In [ ]:
# Display all available water quality parameters from metadata
available_params = [col.replace('obs_', '') for col in df_main_linkages.columns
                    if col.startswith('obs_') and not col.startswith('obs_gwl_') and not any(col.endswith(s) for s in
                    ('_observed', '_LOD2', '_ROS', '_outlier'))]

print(f"Available water quality parameters ({len(available_params)} total):\n")
print(f"{'Rank':<6} {'Parameter':<30} {'Stations':<10} {'Total Obs':>12} "
      f"{'Obs':>12} {'Imputed':>12} {'Outlier':>12}")
print("=" * 96)

param_stats = []
for param in available_params:
    n_stations = (df_main_linkages[f'obs_{param}'] > 0).sum()
    total_obs  = df_main_linkages[f'obs_{param}'].sum()
    n_observed = df_main_linkages[f'obs_{param}_observed'].sum() if f'obs_{param}_observed' in df_main_linkages.columns else np.nan
    n_lod2     = df_main_linkages[f'obs_{param}_LOD2'].sum()     if f'obs_{param}_LOD2'     in df_main_linkages.columns else np.nan
    n_ros      = df_main_linkages[f'obs_{param}_ROS'].sum()      if f'obs_{param}_ROS'      in df_main_linkages.columns else np.nan
    n_outlier  = df_main_linkages[f'obs_{param}_outlier'].sum()  if f'obs_{param}_outlier'  in df_main_linkages.columns else np.nan
    n_imputed  = (n_lod2 + n_ros) if not (np.isnan(n_lod2) or np.isnan(n_ros)) else np.nan
    param_stats.append((param, n_stations, total_obs, n_observed, n_imputed, n_outlier))

# Sort by total observations (descending)
param_stats.sort(key=lambda x: x[2], reverse=True)

def fmt(v, fmt_str):
    return format(v, fmt_str) if not (isinstance(v, float) and np.isnan(v)) else 'N/A'

for i, (param, n_stations, total_obs, n_observed, n_imputed, n_outlier) in enumerate(param_stats, 1):
    print(f"{i:<6} {param:<30} {n_stations:<10,.0f} {fmt(total_obs, '>12,.0f')} "
          f"{fmt(n_observed, '>12,.0f')} {fmt(n_imputed, '>12,.0f')} {fmt(n_outlier, '>12,.0f')}")

#### 6.1.2 Geographic distribution of observations (country-level)

In [ ]:
country_stats = df_main_linkages.groupby('country_name').agg({
    'gwms_id': 'count',
    'total_observations': 'sum',
    'parameters_measured': 'mean'
}).rename(columns={
    'gwms_id': 'n_stations',
    'total_observations': 'total_obs',
    'parameters_measured': 'avg_params'
}).sort_values('total_obs', ascending=False)

print(f"\nTop 20 countries by total observations:")
print(f"{'Rank':<6} {'Country':<30} {'Stations':<12} {'Total Obs':>15} {'Avg Params':>12}")
print("=" * 80)
for i, (country, row) in enumerate(country_stats.head(20).iterrows(), 1):
    print(f"{i:<6} {country:<30} {row['n_stations']:<12,.0f} {row['total_obs']:>15,.0f} {row['avg_params']:>12.1f}")

In [ ]:
selected_country = "United Kingdom"
selected_param = "EC"

# Filter data
country_data = df_main_linkages[
    (df_main_linkages['country_name'] == selected_country) &
    (df_main_linkages[f'obs_{selected_param}'] > 0)
].copy()

if len(country_data) > 0:
    # Create map
    fig, ax = plt.subplots(figsize=(15, 10), subplot_kw={'projection': WGS84_PLATECARREE})

    # Set extent and add features
    buffer = 2
    ax.set_extent([country_data['gwms_lon'].min() - buffer, country_data['gwms_lon'].max() + buffer,
                   country_data['gwms_lat'].min() - buffer, country_data['gwms_lat'].max() + buffer],
                  crs=WGS84_PLATECARREE)

    ax.set_facecolor('white')
    ax.add_geometries(land_geoms, crs=WGS84_PLATECARREE, facecolor='lightgray', alpha=0.3, edgecolor='black', linewidth=1, zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':', zorder=1)

    # Plot stations
    scatter = ax.scatter(country_data['gwms_lon'], country_data['gwms_lat'],
                        c=country_data[f'obs_{selected_param}'], s=50, alpha=0.6,
                        cmap='YlOrRd', edgecolors='black', linewidths=0.5,
                        transform=WGS84_PLATECARREE, zorder=5,
                        vmax=country_data[f'obs_{selected_param}'].quantile(0.99))

    plt.colorbar(scatter, ax=ax, label=f'{selected_param} observations', shrink=0.8)
    ax.set_title(f'{selected_country}: {selected_param} ({len(country_data):,} stations)')
    plt.tight_layout()
    plt.show()

    # Add a summary table
    print(f"\nSummary")
    print("=" * 46)

    country_data['obs_param'] = country_data[f'obs_{selected_param}']
    print(f"Total stations:                {len(country_data):>15,.0f}")
    print(f"Total observations:            {country_data['obs_param'].sum():>15,.0f}")
    print(f"Mean obs per station:          {country_data['obs_param'].mean():>15,.0f}")
    print(f"Median obs per station:        {country_data['obs_param'].median():>15,.0f}")
    print(f"Min obs:                       {country_data['obs_param'].min():>15,.0f}")
    print(f"Max obs:                       {country_data['obs_param'].max():>15,.0f}")

    # Temporal coverage
    country_data['param_duration'] = (
        pd.to_datetime(country_data[f'end_{selected_param}']) -
        pd.to_datetime(country_data[f'start_{selected_param}'])
    ).dt.days / 365.25

### 6.2 Time-series per water quality parameter

In [ ]:
#Make plot for the stations with the most observations for a given parameter
wq_parameter = "TEMP"
obs_col = f'obs_{wq_parameter}'
outlier_col = f'obs_{wq_parameter}_outlier'
param = f'{wq_parameter}'

EXCLUDE_OUTLIERS = True #excludes values flagged as outlier "3" in GROW-Qual, missing values "255" always excluded.

n_top_stations = 3  # Number of stations to plot

if obs_col not in df_main_linkages.columns:
    available_params = [col.replace('obs_', '') for col in df_main_linkages.columns if col.startswith('obs_')]
    print(f"Error: Parameter '{wq_parameter}' not found in dataset")
    print(f"Available parameters: {', '.join(available_params[:20])}...")
else:
    # Rank stations by clean (non-outlier) observation count, not raw total - a station
    # with many observations but mostly outliers isn't actually a good one to feature
    linkages_ranked = df_main_linkages.copy()
    linkages_ranked['clean_obs'] = linkages_ranked[obs_col] - linkages_ranked[outlier_col].fillna(0)

    # Find top N stations by clean observation count for this parameter
    top_stations = linkages_ranked.nlargest(n_top_stations, 'clean_obs')[
        ['gwms_id', obs_col, 'clean_obs', 'country_name', 'gwms_lat', 'gwms_lon',
         f'start_{wq_parameter}', f'end_{wq_parameter}']
    ].copy()

    # Extract data for top stations
    print(f"\nExtracting {wq_parameter} data for top stations...")
    station_data = {}

    for _, station in top_stations.iterrows():
        gwms_id = station['gwms_id']
        data = ds[param].sel(gwms_id=gwms_id)

        flag_var = f"{param}_flag"
        if flag_var in ds.data_vars:
            flags = ds[flag_var].sel(gwms_id=gwms_id)
            exclude_codes = [3, 255] if EXCLUDE_OUTLIERS else [255]
            data = data.where(~flags.isin(exclude_codes))

        if np.sum(~np.isnan(data.values)) > 0:
            station_data[gwms_id] = {
                'data': data,
                'n_obs': int(np.sum(~np.isnan(data.values))),
                'country': station['country_name'],
                'start': station[f'start_{wq_parameter}'],
                'end': station[f'end_{wq_parameter}']
            }

    print(f"Successfully loaded data for {len(station_data)} stations")

    # Plot all top stations
    fig, axes = plt.subplots(len(station_data), 1, figsize=(16, 3*len(station_data)), sharex=True)
    if len(station_data) == 1:
        axes = [axes]

    for idx, (gwms_id, info) in enumerate(station_data.items()):
        data_df = info['data'].to_pandas()

        axes[idx].plot(data_df.index, data_df.values, 'o-',
                       linewidth=0.8, markersize=2, alpha=0.7, color='steelblue')

        # Get units from attributes
        units = info['data'].attrs.get('units', 'unknown')
        axes[idx].set_ylabel(f'{wq_parameter}\n({units})')

        # Add station info as text annotation
        axes[idx].text(0.02, 0.95,
                      f"{gwms_id} | {info['country']} | {info['n_obs']:,} observations",
                      transform=axes[idx].transAxes,
                      verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.5),
                      fontsize=9)

    axes[-1].set_xlabel('Date')
    fig.suptitle(f'Top {n_top_stations} stations by observation count ({wq_parameter})', y=0.995)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'example_wq_data_top3.png'), dpi=300, bbox_inches='tight')
    plt.show()

## 7. Spatio-temporal plots per constituent

In [ ]:
###Per constituent plots
REGION_PATH = data_dir / "Country_Region_Econ_ID.txt"
REGION_LOOKUP = dict(zip(*pd.read_csv(REGION_PATH, sep="\t")[["Country", "Region"]].values.T))
print(f"Region lookup loaded ({len(REGION_LOOKUP)} countries)")

DATA_DIR_OUT = data_dir / "data_per_constituent"
DATA_DIR_OUT.mkdir(parents=True, exist_ok=True)

###plotting orders and colour schemes
REGIONS = ["East Asia & Pacific","Eastern Europe & Central Asia","Latin America & Caribbean","Middle East & North Africa","North America","Southern Asia","Sub-Saharan Africa","Western Europe",]
REGION_COLOURS = {"East Asia & Pacific": "#F8766D","Eastern Europe & Central Asia": "#CD9600","Latin America & Caribbean": "#7CAE00","Middle East & North Africa": "#00BE67","North America": "#00BFC4","Southern Asia": "#00A9FF","Sub-Saharan Africa": "#C77CFF","Western Europe": "#FF61CC","Other": "#BBBBBB",}
STATION_COLOURS = {"Quality only": "#1f78b4","Quality + Quantity": "#e31a1c",}
print("Plotting order and colours defined")

In [ ]:
###processing and mapping functions
def _constituent_spatiotemporal(param, distance_threshold_m=100.0, exclude_outliers=True):
    if param not in ds.data_vars:
        return None, None

    data = ds[param]
    flag_var = f"{param}_flag"
    if flag_var in ds.data_vars:
        flags = ds[flag_var]
        exclude_codes = [3, 255] if exclude_outliers else [255]  #always drop missing (255), outlier (3) only if requested
        usable = data.where(~flags.isin(exclude_codes))
    else:
        usable = data

    notna = usable.notnull()

    obs_per_site = notna.sum(dim="time").load()
    site_ids = ds.gwms_id.values[obs_per_site.values > 0]
    if len(site_ids) == 0:
        return None, None

    meta_cols = ["gwms_id", "country_name", "gwms_lat", "gwms_lon", "grow_id", "grow_distance_m"]
    meta = (df_main_linkages[meta_cols].drop_duplicates("gwms_id").set_index("gwms_id").reindex(site_ids))
    meta["has_grow_thresh"] = (
        meta["grow_id"].notna() &
        (meta["grow_distance_m"].fillna(np.inf) <= distance_threshold_m)
    )
    meta["Region"] = meta["country_name"].map(REGION_LOOKUP)
    meta["Region_grouped"] = meta["Region"].where(meta["Region"].isin(REGIONS), "Other")
    meta["category"] = np.where(meta["has_grow_thresh"], "Quality + Quantity", "Quality only")
    site_df = meta.reset_index().rename(columns={"index": "gwms_id"})

    monthly_obs = notna.sel(gwms_id=site_ids).resample(time="MS").sum().to_pandas()

    obs_long = monthly_obs.stack().rename("n_obs").reset_index()
    obs_long.columns = ["gwms_id", "dates", "n_obs"]
    obs_long = obs_long[obs_long["n_obs"] > 0].copy()
    obs_long["year"] = obs_long["dates"].dt.year
    obs_long["month"] = obs_long["dates"].dt.month
    obs_long = obs_long.merge(site_df[["gwms_id", "Region_grouped"]], on="gwms_id", how="left")

    ts_month = (obs_long.groupby(["Region_grouped", "year", "month"])[["n_obs"]]
                .sum().reset_index())
    ts_month["dates"] = pd.to_datetime(dict(year=ts_month["year"], month=ts_month["month"], day=1))

    return site_df, ts_month


def _collapse_to_annual(ts_month):
    out = ts_month.groupby(["Region_grouped", "year"])[["n_obs"]].sum().reset_index()
    out["dates"] = pd.to_datetime(out["year"].astype(str) + "-01-01")
    return out


def _cache_paths(constituent):
    return (DATA_DIR_OUT / f"{constituent}_site_info.csv",
            DATA_DIR_OUT / f"{constituent}_ts_month.csv")


def _save_processed_data(constituent, site_df, ts_month):
    site_path, ts_path = _cache_paths(constituent)
    site_df.to_csv(site_path, index=False)
    ts_month.drop(columns="dates").to_csv(ts_path, index=False)


def _load_processed_data(constituent):
    site_path, ts_path = _cache_paths(constituent)
    if not (site_path.exists() and ts_path.exists()):
        return None, None
    site_df = pd.read_csv(site_path)
    ts_month = pd.read_csv(ts_path)
    ts_month["dates"] = pd.to_datetime(dict(year=ts_month["year"], month=ts_month["month"], day=1))
    return site_df, ts_month


###plotting functions
def _plot_site_map(ax, site_df, title):
    ax.set_facecolor("white")
    ax.add_geometries(land_geoms, crs=WGS84_PLATECARREE, facecolor="lightgray", edgecolor="black", linewidth=0.1, zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5, zorder=1)

    for category, colour in STATION_COLOURS.items():
        sub = site_df[site_df["category"] == category]
        if not sub.empty:
            ax.scatter(sub["gwms_lon"], sub["gwms_lat"], transform=WGS84_PLATECARREE,
                       s=1, alpha=0.8, color=colour, edgecolors="none", label=category, zorder=5)

    ax.set_global()
    ax.set_title(title, fontsize=15, fontweight="bold", pad=10)
    ax.legend(loc="center", bbox_to_anchor=(0.5, 0.15), fontsize=11, markerscale=6,
              frameon=True, facecolor="white", framealpha=0.7)


def _plot_regional_timeseries(ax, ts_df, year_min, year_max):
    pivot = (ts_df.pivot_table(index="dates", columns="Region_grouped", values="n_obs", aggfunc="sum")
             .fillna(0).sort_index())
    region_order = [r for r in REGIONS if r in pivot.columns] + (["Other"] if "Other" in pivot.columns else [])
    stack_order = region_order[::-1]
    pivot = pivot[stack_order]
    colours = [REGION_COLOURS[r] for r in stack_order]

    polys = ax.stackplot(pivot.index, pivot.T.values, labels=stack_order, colors=colours, alpha=0.9)
    xmin, xmax = pd.Timestamp(f"{year_min}-01-01"), pd.Timestamp(f"{year_max}-12-31")
    ax.set_xlim(xmin, xmax)
    ax.set_ylabel("Number of observations")
    ax.set_xlabel("Year")
    ax.margins(y=0)

    handles = dict(zip(stack_order, polys))
    ax.legend([handles[r] for r in region_order], region_order,
              loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4, fontsize=11, frameon=False)


def plot_constituent(constituent, plot_monthly=True, year_min=1980, year_max=2025,
                      distance_threshold_m=100.0, exclude_outliers=True,
                      save=True, save_data=True, use_cache=False):
    site_df, ts_month = (_load_processed_data(constituent) if use_cache else (None, None))
    if site_df is None:
        site_df, ts_month = _constituent_spatiotemporal(constituent, distance_threshold_m, exclude_outliers)
        if site_df is None:
            print(f"No data available for {constituent}.")
            return None, None
        if save_data:
            _save_processed_data(constituent, site_df, ts_month)
    else:
        print(f"  Loaded cached processed data for {constituent}")

    title = constituent
    match = var_metadata.loc[var_metadata["variable"] == constituent, "description"]
    if len(match) and pd.notna(match.iloc[0]):
        title = f"{match.iloc[0]} ({constituent})"

    ts_plot = ts_month if plot_monthly else _collapse_to_annual(ts_month)

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 2], hspace=0.05)
    ax_map = fig.add_subplot(gs[0], projection=WGS84_ROBINSON)
    ax_ts = fig.add_subplot(gs[1])

    _plot_site_map(ax_map, site_df, title)
    _plot_regional_timeseries(ax_ts, ts_plot, year_min, year_max)

    plt.tight_layout()
    if save:
        plt.savefig(os.path.join(plot_dir, f"{constituent}_spatiotemporal.png"), dpi=300, bbox_inches="tight")
    plt.show()

    return site_df, ts_month

print("Processing and plotting functions for per-constituent plots loaded!")

In [ ]:
###Run per-constituent plots
plot_constituent("EC", plot_monthly=True, year_min=1980, year_max=2025, exclude_outliers=True, save=True, save_data=True, use_cache=True); #process and plot defined constituent. If use_cache is True, function will attempt to use pre-processed .csv output to make these plots, otherwise the required plotting data will be reprocessed directly from the .zarr